# 🐕 Stray Dogs Tunisia — Système Complet
**ESPRIT — 3ème année cycle Ingénieur IA | Axe Déchets & Urbanisme**

## Comment utiliser ce notebook ?
1. **Cellule SETUP** : exécuter une seule fois pour préparer l'environnement
2. **Cellule ENTRAÎNEMENT** : générer les données synthétiques et entraîner les modèles
3. **Cellule INPUT** : remplir les informations de ton quartier
4. **Cellule RUN** : lancer le pipeline → obtenir les 3 solutions
5. **Cellules RÉSULTATS** : visualiser chaque solution en détail

---
| Solution | Description | Approche technique |
|---|---|---|
| S1 | Estimation densité canine | Random Forest + Gradient Boosting (ensemble) |
| S2 | Recommandation & placement des bennes | Formule calibrée + OSM Overpass + scoring géospatial |
| S3 | Identification Feeding Zones | Score AHP multicritère |
| CV | Détection infractions nourrissage | YOLOv8n + Optical Flow + Backtracking vidéo |

## 📦 ÉTAPE 1 — Setup (exécuter une seule fois)

In [ ]:
import sys, os
from pathlib import Path

# Ajouter src/ au path
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
SRC_DIR      = PROJECT_ROOT / 'src'
sys.path.insert(0, str(SRC_DIR))
sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display, Image, HTML

from config import ROOT, DATA_SYN, MDL_DIR, VIZ_DIR
from pipeline.pipeline import run_pipeline, load_models
from utils.osm_fetcher import get_district_data, extract_elements_from_png
from utils.visualizer import plot_full_dashboard, plot_interactive_map_html

print('✅ Imports OK')
print(f'   Projet root : {PROJECT_ROOT}')

## 🧠 ÉTAPE 2 — Génération données synthétiques + Entraînement

In [ ]:
from models.generate_synthetic_data import generate_full_dataset

# Génération du dataset (1200 secteurs synthétiques, 24 gouvernorats)
print('Génération du dataset synthétique...')
df_synthetic = generate_full_dataset()

print(f'\nAperçu du dataset :')
display(df_synthetic[['gouvernorat','zone_type','population','nb_food_poi','nb_bennes_org','nb_chiens','risk_class']].head(10))

In [ ]:
from models.train_models import run as train_all

# Entraînement de tous les modèles
# Random Forest Regressor + Random Forest Classifier + Gradient Boosting
rf_reg, gb_reg, rf_clf, scaler = train_all(evaluate=True)

# Afficher les résultats d'entraînement
img_path = VIZ_DIR / 'training_results.png'
if img_path.exists():
    fig, ax = plt.subplots(figsize=(20, 12), facecolor='#0d1117')
    ax.imshow(mpimg.imread(img_path))
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    print('✅ Modèles entraînés et sauvegardés dans models/trained/')

## 🏙️ ÉTAPE 3 — INPUT : Définir le quartier

### Option A — Par nom (OpenStreetMap récupère tout automatiquement)
### Option B — Manuellement (si pas d'accès internet ou quartier inconnu)
### Option C — Depuis un PNG (carte avec éléments colorés)

In [ ]:
# ═══════════════════════════════════════════════════════════
#  ▶ REMPLIR ICI — Informations sur le quartier
# ═══════════════════════════════════════════════════════════

# ── Informations de base ────────────────────────────────────
DISTRICT_NAME  = 'La Marsa'          # Nom du quartier/ville
COUNTRY        = 'Tunisia'           # Pays

# ── Si tu connais la bbox (optionnel, sinon OSM la trouve) ──
MANUAL_BBOX = None  # ou dict ex: {'lat_min': 36.862, 'lat_max': 36.930, 'lon_min': 10.295, 'lon_max': 10.350}

# ── Informations complémentaires ────────────────────────────
POPULATION    = 92987     # Nombre d'habitants (RGPH 2014)
NB_MENAGES    = 26062     # Nombre de ménages
SURFACE_KM2   = 12.5      # Surface en km²
ZONE_TYPE     = 'mixed'   # residential | commercial | mixed | beach | periphery | ...
N_ZONES       = 7         # Nombre de feeding zones à identifier

# ── PNG de carte (optionnel) ────────────────────────────────
PNG_PATH = None  # ex: '../data/raw/carte_lamarsa.png'
#
# Convention couleurs sur le PNG :
#   Rouge  = bennes organiques
#   Bleu   = bennes plastique
#   Orange = restaurants/cafés
#   Vert   = espaces verts/parcs
#   Jaune  = écoles

print(f'✅ Configuration : {DISTRICT_NAME}')
print(f'   Population  : {POPULATION:,}')
print(f'   Ménages     : {NB_MENAGES:,}')
print(f'   Surface     : {SURFACE_KM2} km²')
print(f'   Zone        : {ZONE_TYPE}')
print(f'   PNG         : {PNG_PATH or "Non fourni → OSM"}')

## 🗺️ ÉTAPE 4 — Récupération des données OSM

In [ ]:
# Récupération des données du quartier
print(f'Récupération des données pour : {DISTRICT_NAME}...')

district_data = get_district_data(
    district_name=DISTRICT_NAME,
    bbox=MANUAL_BBOX,
    country=COUNTRY,
)

BBOX      = district_data['bbox']
food_poi  = district_data['food_poi']
schools   = district_data['schools']
parks     = district_data['parks']

print(f'\n✅ Données récupérées :')
print(f'   Bbox       : lat [{BBOX["lat_min"]:.3f}, {BBOX["lat_max"]:.3f}]')
print(f'   Food POI   : {len(food_poi)}')
print(f'   Écoles     : {len(schools)}')
print(f'   Parcs      : {len(parks)}')

In [ ]:
# ── Bennes : depuis PNG ou saisie manuelle ───────────────────

# Option A : extraction depuis PNG
bins_from_png = {}
if PNG_PATH and Path(PNG_PATH).exists():
    bins_from_png = extract_elements_from_png(PNG_PATH, BBOX)
    print(f'Éléments extraits du PNG : {sum(len(v) for v in bins_from_png.values())} points')
    
    # Construire le DataFrame des bennes depuis le PNG
    bins_rows = []
    for lat, lon in bins_from_png.get('bins_org', []):
        bins_rows.append({'lat': lat, 'lon': lon, 'type': 'organique'})
    for lat, lon in bins_from_png.get('bins_plas', []):
        bins_rows.append({'lat': lat, 'lon': lon, 'type': 'plastique'})
    BINS_DF = pd.DataFrame(bins_rows)
    
    # Enrichir food_poi et schools depuis le PNG
    if bins_from_png.get('food_poi'):
        extra_food = pd.DataFrame(bins_from_png['food_poi'], columns=['lat','lon'])
        extra_food['type'] = 'restaurant'; extra_food['category'] = 'food'; extra_food['source'] = 'png'
        food_poi = pd.concat([food_poi, extra_food], ignore_index=True)
    if bins_from_png.get('schools'):
        extra_schools = pd.DataFrame(bins_from_png['schools'], columns=['lat','lon'])
        extra_schools['category'] = 'school'; extra_schools['source'] = 'png'
        schools = pd.concat([schools, extra_schools], ignore_index=True)

else:
    # Option B : bennes simulées (distribution réaliste)
    print('Pas de PNG → génération de bennes simulées...')
    from src.utils.bin_generator import generate_bins_for_bbox
    BINS_DF = generate_bins_for_bbox(BBOX, POPULATION)
    print(f'   {len(BINS_DF)} bennes générées')

print(f'\n✅ Bennes : {len(BINS_DF)} total')
if not BINS_DF.empty:
    print(BINS_DF['type'].value_counts().to_string())

## 🚀 ÉTAPE 5 — Lancement du pipeline (3 solutions)

In [ ]:
# Chargement des modèles entraînés
models = load_models()

# Pipeline complet → 3 solutions
results = run_pipeline(
    district_name = DISTRICT_NAME,
    population    = POPULATION,
    nb_menages    = NB_MENAGES,
    surface_km2   = SURFACE_KM2,
    bbox          = BBOX,
    bins_df       = BINS_DF,
    food_poi_df   = food_poi,
    schools_df    = schools,
    parks_df      = parks,
    zone_type     = ZONE_TYPE,
    n_zones       = N_ZONES,
    models        = models,
    verbose       = True,
)

s1 = results['s1']
s2 = results['s2']
s3 = results['s3']

print('\n✅ Pipeline terminé.')

## 📊 ÉTAPE 6 — Résultats

In [ ]:
# ─── SOLUTION 1 : Densité canine ───────────────────────────
print('═'*55)
print('  SOLUTION 1 — ESTIMATION DENSITÉ CANINE')
print('═'*55)
print(f'''
  Quartier       : {DISTRICT_NAME}
  Population     : {POPULATION:,} habitants
  ──────────────────────────────────────────
  Chiens estimés : {s1["nb_chiens"]:,}
  Ratio          : {s1["details"]["ratio_chien_habitant"]}
  Niveau risque  : {s1["risk_label"]}
  Confiance      : {s1["confidence"]*100:.0f}%
  Modèle         : {s1["source"]}
  ──────────────────────────────────────────
  Contribution POI alimentaires : +{s1["details"]["contribution_food_poi"]} chiens
  Contribution bennes org       : +{s1["details"]["contribution_bennes"]} chiens
  Coeff zone ({ZONE_TYPE})     : ×{s1["details"]["zone_coeff"]}
''')

In [ ]:
# ─── SOLUTION 2 : Bennes recommandées ───────────────────────
print('═'*60)
print('  SOLUTION 2 — RECOMMANDATION & PLACEMENT DES BENNES')
print('═'*60)

st  = s2['stats']
bci = s2.get('bin_count_info', {})
rec = s2.get('recommended_bins', None)

print(f'''
  ── Formule de recommandation ─────────────────────────────
  {st.get("formule", "N/A")}

  ── Résumé ───────────────────────────────────────────────
  Bennes recommandées  : {st.get("nb_recommande", "—")}
  Bennes effectivement placées : {st.get("bennes_placees", "—")}
  Intersections OSM trouvées   : {st.get("intersections_osm", "—")}
  Restaurants détectés         : {st.get("nb_restaurants", "—")}
  Cafés détectés               : {st.get("nb_cafes", "—")}
  Points noirs (bennes critiques) : {st.get("black_spots", "—")}

  ── Décomposition des contributions ──────────────────────''')

contrib = st.get("contributions", {})
for k, v in contrib.items():
    print(f'  {k:<20s} : +{v} bennes')

print()
if rec is not None and not rec.empty:
    print(f'\n  Bennes recommandées (top 10) :')
    display(rec[['lat','lon','score','type','source']].head(10).round(5))
else:
    print('  (aucune benne placée — vérifier la bbox ou les paramètres)')

In [ ]:
# ─── SOLUTION 3 : Feeding Zones ───────────────────────────
print('═'*55)
print('  SOLUTION 3 — FEEDING ZONES')
print('═'*55)

fz = s3['feeding_zones']
if not fz.empty:
    print(f'  {len(fz)} zones identifiées\n')
    display(fz[['zone_id','lat','lon','score_pct','rayon_m']].round(4))
else:
    print('  Aucune zone (grille vide — vérifier la bbox)')

In [ ]:
# ─── Dashboard visuel ─────────────────────────────────────
print('Génération du dashboard...')

out_path = plot_full_dashboard(
    district_name = DISTRICT_NAME,
    bbox          = BBOX,
    s1_result     = s1,
    s2_result     = s2,
    s3_result     = s3,
    bins_df       = BINS_DF,
    food_poi_df   = food_poi,
    schools_df    = schools,
    parks_df      = parks,
    save          = True,
)

# Afficher dans le notebook
fig, ax = plt.subplots(figsize=(22, 15), facecolor='#0d1117')
ax.imshow(mpimg.imread(out_path))
ax.axis('off')
plt.tight_layout()
plt.show()
print(f'✅ Dashboard : {out_path}')

In [ ]:
# ─── Carte interactive HTML (si folium installé) ──────────
html_path = plot_interactive_map_html(
    district_name = DISTRICT_NAME,
    bbox          = BBOX,
    s1            = s1,
    s2            = s2,
    s3            = s3,
    bins_df       = BINS_DF,
    food_poi_df   = food_poi,
    schools_df    = schools,
    parks_df      = parks,
)

if html_path:
    print(f'✅ Ouvrir dans le navigateur : {html_path}')
    display(HTML(f'<a href="{html_path}" target="_blank">🗺️ Ouvrir la carte interactive</a>'))

In [ ]:
# ─── Export des résultats en CSV ──────────────────────────
from config import DATA_OUT
slug = DISTRICT_NAME.lower().replace(' ', '_')

rec_bins = s2.get('recommended_bins', None)
if rec_bins is not None and not rec_bins.empty:
    rec_bins.to_csv(DATA_OUT / f'{slug}_bennes_recommandees.csv', index=False)
    print(f'✅ Bennes exportées : {len(rec_bins)} lignes')

if not s3['feeding_zones'].empty:
    s3['feeding_zones'].to_csv(DATA_OUT / f'{slug}_feeding_zones.csv', index=False)

import json
st = s2.get('stats', {})
summary = {
    'district':            DISTRICT_NAME,
    'population':          POPULATION,
    'chiens_estimes':      s1['nb_chiens'],
    'niveau_risque':       s1['risk_label'],
    'confidence':          s1['confidence'],
    'bennes_recommandees': st.get('nb_recommande', 0),
    'bennes_placees':      st.get('bennes_placees', 0),
    'intersections_osm':   st.get('intersections_osm', 0),
    'black_spots':         st.get('black_spots', 0),
    'feeding_zones':       len(s3['feeding_zones']),
}
with open(DATA_OUT / f'{slug}_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print('✅ Résultats exportés dans data/outputs/')
print(json.dumps(summary, indent=2, ensure_ascii=False))